In [1]:
# ============================================
# California Coastal Conditions Pipeline
# Stage 3 of 4: Pull current weather for each lighthouse location
# Author: Brittany Blessie
# Description: Calls the OpenWeatherMap API for every California lighthouse coordinate,
#              flattens the nested JSON, and applies labeled transformation steps to produce
#              a clean weather dataset.
# ============================================

In [2]:
import pandas as pd
import requests

## Explore: Test the API With One Call
Make a single API call for one lighthouse to see the JSON structure before looping over all 50 locations.

In [3]:
# Single test call to OpenWeatherMap for one location, to see the JSON structure before looping

# Read the API key from a local file that is not committed to the repo.
# Create api_key.txt beside this notebook and paste your key into it.
# Free keys: https://openweathermap.org/api
with open("api_key.txt") as f:
    api_key = f.read().strip()

# Use one lighthouse's coordinates as a test (Alcatraz: 37.826250, -122.422167)
test_lat = 37.826250
test_lon = -122.422167

url = "https://api.openweathermap.org/data/2.5/weather"
params = {"lat": test_lat, "lon": test_lon, "appid": api_key, "units": "imperial"}

response = requests.get(url, params=params)
print("Status code:", response.status_code)
response.json()

Status code: 200


{'coord': {'lon': -122.4222, 'lat': 37.8263},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 68.67,
  'feels_like': 68.2,
  'temp_min': 65.79,
  'temp_max': 71.62,
  'pressure': 1016,
  'humidity': 63,
  'sea_level': 1016,
  'grnd_level': 1012},
 'visibility': 10000,
 'wind': {'speed': 10, 'deg': 270, 'gust': 15.99},
 'clouds': {'all': 100},
 'dt': 1788296529,
 'sys': {'type': 2,
  'id': 2017837,
  'country': 'US',
  'sunrise': 1788269997,
  'sunset': 1788316788},
 'timezone': -25200,
 'id': 5391959,
 'name': 'San Francisco',
 'cod': 200}

**Observation:** The loop returned weather for all 50 lighthouses in a clean table of six columns. The values match real geography, with the northern coast cooler than San Diego, which confirms the coordinates were sent correctly.

## Step 1: Pull and Flatten the API Data
Read the cleaned lighthouse coordinates, then loop through all 50 locations, call the API for each, and flatten the nested JSON into a table of temperature, humidity, wind speed, and description.

In [4]:
# Read the cleaned lighthouse coordinates from the previous stage
lighthouses = pd.read_csv("lighthouses_clean.csv")
print(lighthouses.shape)
lighthouses.head()

# Step #1: Loop through all 50 lighthouse locations and pull current weather from the API.
# For each lighthouse, call the API with its coordinates, then reach into the nested JSON
# to grab temperature, humidity, wind speed, and description, and collect them into a list.

weather_records = []

for index, row in lighthouses.iterrows():
    lat = row["Latitude"]
    lon = row["Longitude"]
    params = {"lat": lat, "lon": lon, "appid": api_key, "units": "imperial"}
    response = requests.get(url, params=params)
    data = response.json()

    weather_records.append({
        "Latitude": lat,
        "Longitude": lon,
        "Temperature_F": data["main"]["temp"],
        "Humidity": data["main"]["humidity"],
        "Wind_Speed": data["wind"]["speed"],
        "Description": data["weather"][0]["description"]
    })

weather = pd.DataFrame(weather_records)
print(weather.shape)
weather.head()

(50, 4)
(50, 6)


,Latitude,Longitude,Temperature_F,Humidity,Wind_Speed,Description
0,37.826250,-122.422167,68.67,63,10.00,overcast clouds
1,34.015827,-119.359548,73.51,70,7.63,clear sky
2,37.108300,-122.337800,71.58,54,6.29,overcast clouds
3,32.686389,-117.232500,79.74,68,5.01,clear sky
4,41.744094,-124.203099,61.45,88,11.50,moderate rain


## Step 2: Check for Missing Values
Count missing values in every column to confirm the API returned complete data before any further cleaning.

In [5]:
# Step #2: Check for missing values in the weather data.
weather.isnull().sum()

Latitude         0
Longitude        0
Temperature_F    0
Humidity         0
Wind_Speed       0
Description      0
dtype: int64

**Observation:** Every column shows zero missing values, so the API returned complete data for all 50 locations. No rows need to be dropped for missing weather.

## Step 3: Check for Duplicate Rows
Count exact duplicate rows, since several lighthouses sit close together and could return identical weather from the same nearby station.

In [6]:
# Step #3: Check for duplicate rows in the weather data.
weather.duplicated().sum()

np.int64(0)

**Observation:** The check found zero duplicate rows. Even though some lighthouses sit close together, each returned distinct weather, so no rows need to be removed.

## Step 4: Check Data Types
Confirm each column has the correct type, so temperature, humidity, and wind speed are numbers and description is text before any further use.

In [7]:
# Step #4: Check the data type of each column.
weather.dtypes

Latitude         float64
Longitude        float64
Temperature_F    float64
Humidity           int64
Wind_Speed       float64
Description       object
dtype: object

**Observation:** All columns have the correct type. Temperature, wind speed, and coordinates are float numbers, humidity is a whole number, and description is text. No type conversion is needed.

## Step 5: Check for Impossible Values
Use summary statistics to confirm the weather values fall in realistic ranges, since impossible values like negative humidity or negative wind would signal bad data.

In [8]:
# Step #5: Check summary statistics to spot any impossible weather values.
weather.describe()

,Latitude,Longitude,Temperature_F,Humidity,Wind_Speed
count,50.000000,50.000000,50.000000,50.000000,50.000000
mean,37.014796,-121.497327,69.788800,67.360000,10.039400
std,2.495800,2.122180,4.831018,10.918548,4.285039
min,32.665071,-124.406031,61.020000,35.000000,1.010000
25%,34.480829,-122.517075,67.170000,63.000000,7.495000
50%,37.787050,-122.247141,69.090000,67.000000,10.000000
75%,38.067846,-120.099533,72.102500,71.000000,11.990000
max,41.836330,-117.232500,79.740000,90.000000,20.710000


**Observation:** All values fall in realistic ranges. Humidity stays between 0 and 100, wind speed is never negative, temperatures are reasonable for the California coast, and the coordinates match California's geographic range. No impossible values were found.

## Step 6: Clean Up the Description Text
Capitalize the first letter of each weather description so the final dataset reads more cleanly.

In [9]:
# Step #6: Capitalize the first letter of each weather description for readability.
weather["Description"] = weather["Description"].str.capitalize()
weather.head()

,Latitude,Longitude,Temperature_F,Humidity,Wind_Speed,Description
0,37.826250,-122.422167,68.67,63,10.00,Overcast clouds
1,34.015827,-119.359548,73.51,70,7.63,Clear sky
2,37.108300,-122.337800,71.58,54,6.29,Overcast clouds
3,32.686389,-117.232500,79.74,68,5.01,Clear sky
4,41.744094,-124.203099,61.45,88,11.50,Moderate rain


**Observation:** The descriptions now start with a capital letter, so the final dataset reads more cleanly. This is a formatting change only and does not affect the values.

## Final Cleaned Weather Dataset

In [10]:
# Final cleaned weather dataset (all 50 lighthouse locations)
weather

,Latitude,Longitude,Temperature_F,Humidity,Wind_Speed,Description
0,37.826250,-122.422167,68.67,63,10.00,Overcast clouds
1,34.015827,-119.359548,73.51,70,7.63,Clear sky
2,37.108300,-122.337800,71.58,54,6.29,Overcast clouds
3,32.686389,-117.232500,79.74,68,5.01,Clear sky
4,41.744094,-124.203099,61.45,88,11.50,Moderate rain
5,40.439906,-124.406031,61.79,81,13.71,Overcast clouds
6,38.067816,-122.213832,70.90,58,11.01,Overcast clouds
7,37.963233,-122.433643,69.46,67,10.00,Overcast clouds
8,37.698966,-123.001651,66.94,72,8.68,Broken clouds
9,37.810560,-122.477333,68.52,62,10.00,Overcast clouds


## Data Notes and Limitations

Here is what I did to the API data and the ethics behind it. I pulled current weather for all 50 lighthouse coordinates from the OpenWeatherMap API, flattened the nested JSON into a table of temperature, humidity, wind speed, and description, then checked for missing values and duplicates, confirmed the data types, checked the values were realistic, and capitalized the descriptions so they read cleanly. For legal and regulatory guidelines, OpenWeatherMap lets you use this data for free within its rate limits, and the data is just weather, so there is nothing personal or sensitive in it. The biggest risk is that this weather is a single snapshot from the moment I called the API, so anyone using it should know it is one point in time, not the usual or historical conditions at each spot. There is also the fact that the API pulls from the nearest reporting station, so a reading might reflect conditions a little off from the exact lighthouse. For assumptions, I took the API values as accurate and assumed the nearest station is close enough to stand in for each location. For sourcing, the data came straight from the official OpenWeatherMap API, and I checked it by confirming the numbers matched real California geography, with the north coast cooler and the south warmer, which is what I expected. It was collected ethically, since I used a free authorized key and stayed within the rate limit so I was not overloading the service. To handle the risks, I noted that the weather is a point-in-time snapshot, kept my API key private, and would always include the collection date so no one mistakes it for long-term climate data.

In [11]:
# Save the cleaned weather data for the merge stage
weather.to_csv("weather_clean.csv", index=False)